# 02 · RNA + DNA QC (ResolveOME, XDP G&T plate)

Builds one AnnData with:
- **RNA**: BJ-Expression count matrices (HTSeq + Salmon) as layers, plus per-cell RNA QC and metadata
- **DNA**: BJ-DNA-QC metrics (subsampled → used for QC only)
- one pass/fail verdict per library (`rna_pass`, `dna_pass`) and a combined one (`both_pass`, `libs_pass`)
- MapMyCells cell types (Allen basal ganglia atlas)

**Cells vs libraries.** Each cell (`cell_00` … `cell_47`) has three libraries in different wells:
a DNA library, an RNA library and a spatial-barcode library. The link comes from the plate map
(`XDP_GandT_plate_map_cells.csv`, from WALKUP-21131). Cells are always named by `cell_id`;
wells only appear in the `lib_*` columns.

**Column prefixes in `adata.obs`**

| prefix | source |
|---|---|
| `lib_` | plate map: DNA / RNA / spatial wells, indices, Trekker tile |
| `rna_` | MultiQC general stats (fastp / STAR / Salmon) |
| `rna_qm_` | Qualimap % exonic / intergenic |
| `rna_mt_` | mitochondrial counts |
| `rna_ct_` | SingleR cell type + cell cycle (BioSkryb) |
| `rna_htseq__` | HTSeq summary rows (`__no_feature`, `__ambiguous`, …) |
| `dna_` | BJ-DNA-QC metrics |
| `dna_bioskryb_` | BioSkryb per-metric verdicts (0/1) |
| `mmc_` | MapMyCells (Allen basal ganglia atlas) |

In [1]:
import os, json, tempfile
import numpy as np
import pandas as pd
import anndata as ad

## 0. Paths, plate map and helpers

In [2]:
# ---- inputs ----
RNA_RES = "/home/gdallagl/myworkdir/XDP/data/XDP/G_and_T/rna_libs/results_BJ-Expression"
HTSEQ   = f"{RNA_RES}/secondary_analyses/quantification_htseq"
SALMON  = f"{RNA_RES}/secondary_analyses/quantification_salmon"

DNA_RES = "/home/gdallagl/myworkdir/XDP/data/XDP/G_and_T/dna_libs/results_BJ-DNA-QC"
DNA_QC  = f"{DNA_RES}/tertiary_analyses/qc_plots"

PLATE_MAP = "/home/gdallagl/myworkdir/XDP/data/XDP/G_and_T/XDP_GandT_plate_map_cells.csv"

# ---- MapMyCells (Allen basal ganglia atlas) ----
MMC_MARKERS = os.getenv("ALLEN_ATLAS_BG_MMC_MARKER_JSON")   # marker-gene lookup
MMC_STATS   = os.getenv("ALLEN_ATLAS_BG_MMC_STAT_H5")       # precomputed reference stats
MMC_DIR     = "/home/gdallagl/myworkdir/XDP/data/_tmp/G_and_T/mapmycells"

# ---- output ----
OUT_H5AD = "/home/gdallagl/myworkdir/XDP/data/XDP/G_and_T/rna_libs/adata_RNA_DNA-QC.h5ad"

In [3]:
# plate map: one row per cell -> which DNA / RNA / spatial well it was put in
plate = pd.read_csv(PLATE_MAP).set_index("cell_id")
display(plate.head(3))

# lookups: well -> cell_id
RNA_WELL_TO_CELL = pd.Series(plate.index, index=plate["rna_well"])
DNA_WELL_TO_CELL = pd.Series(plate.index, index=plate["dna_well"])

,dna_well,dna_index,dna_i5,dna_i7,rna_well,rna_index,rna_i5,rna_i7,spatial_well,spatial_i5,spatial_i7,trekker_tile
cell_id,,,,,,,,,,,,
cell_00,A2,C-01A,AGACCCTATA,TGGCAGGGCG,A7,C-07A,CGGACCATAC,CTTCATCTTA,A12,ATCGATCTGG,AAGTCAATGG,U0059_015
cell_01,B2,C-02A,TAACTGAATC,AAAGGCTAGA,B7,C-08A,CGAACATATA,ATTGTGAATA,B12,GATTCAGGCG,AAGTCAATGG,U0059_015
cell_02,C2,C-01B,AGTACAGAAT,GAATTATATC,C7,C-07B,CCCAGTCATC,TGGCCGGGAT,C12,CCACATATCG,AAGTCAATGG,U0059_015


In [4]:
def cell_from_rna(names):
    """'2535GJLT4-1-0457512804-G8-RNA' -> RNA well 'G8' -> 'cell_xx'  (NaN if not a cell)"""
    well = pd.Index(names).astype(str).str.extract(r"-([A-P]\d+)-RNA")[0]
    return well.map(RNA_WELL_TO_CELL).values


def cell_from_dna(names):
    """'2535GJLT4-1-0457512804-K2-DNA' -> DNA well 'K2' -> 'cell_xx'  (NaN if not a cell)"""
    well = pd.Index(names).astype(str).str.extract(r"-([A-P]\d+)-DNA")[0]
    return well.map(DNA_WELL_TO_CELL).values


def cells_by_genes(m):
    """genes x cells table -> cells x genes; drops columns that aren't cells and
    merges duplicated gene rows by summing them."""
    m = m.loc[:, pd.notna(m.columns)]
    m = m.groupby(level=0).sum()
    return m.T

# 1. RNA (BJ-Expression)

## 1.1 Count matrices

In [5]:
# ---- HTSeq: rows are 'ENSGxxx_SYMBOL', plus summary rows starting with '__' ----
htseq_raw = pd.read_csv(f"{HTSEQ}/matrix_gene_counts_starhtseq.txt", sep="\t", index_col=0)
htseq_raw.columns = cell_from_rna(htseq_raw.columns)

# summary rows (__no_feature, __ambiguous, ...) are per-cell QC, not genes
is_summary = htseq_raw.index.str.startswith("__")
htseq_summary = htseq_raw[is_summary].T                     # cells x summary rows
htseq_summary = htseq_summary.loc[pd.notna(htseq_summary.index)]

# split 'ENSG00000111640.15_GAPDH' -> gene_id, gene_symbol (n=1 keeps symbols containing '_')
genes_raw = htseq_raw[~is_summary]
id_sym = genes_raw.index.str.split("_", n=1)
gene_table = pd.DataFrame({"gene_symbol": id_sym.str[1].values}, index=id_sym.str[0].values)
gene_table = gene_table[~gene_table.index.duplicated()]     # one row per gene_id

genes_raw.index = id_sym.str[0]
htseq = cells_by_genes(genes_raw)                           # cells x gene_id (all HTSeq genes)
display(htseq.iloc[:3, :5])

gene_id_gene_symbol,ENSG00000000419.14,ENSG00000000457.14,ENSG00000000938.13,ENSG00000000971.16,ENSG00000001084.13
cell_00,0,0,0,0,0
cell_16,0,0,0,0,0
cell_01,0,0,0,0,0


In [6]:
# ---- Salmon gene-level matrices: rows are gene_id, plus a 'gene_symbol' column ----
def load_salmon(file_name):
    m = pd.read_csv(f"{SALMON}/{file_name}", sep="\t", index_col=0)
    m = m.drop(columns="gene_symbol", errors="ignore")
    m.columns = cell_from_rna(m.columns)
    return cells_by_genes(m)

salmon_counts       = load_salmon("matrix_gene_counts_salmon.tsv")       # estimated counts
salmon_tpm          = load_salmon("matrix_gene_tpm_salmon.tsv")          # TPM
salmon_lengthscaled = load_salmon("matrix_gene_length_tpm_salmon.tsv")   # length-scaled TPM (for tximport / DE)

for name, m in [("htseq", htseq), ("salmon_counts", salmon_counts),
                ("salmon_tpm", salmon_tpm), ("salmon_lengthscaled", salmon_lengthscaled)]:
    print(f"{name:20s} {m.shape[0]} cells x {m.shape[1]} genes")

htseq                43 cells x 12899 genes
salmon_counts        34 cells x 12293 genes
salmon_tpm           34 cells x 12293 genes
salmon_lengthscaled  34 cells x 12293 genes


## 1.2 Per-cell MultiQC stats (fastp / STAR / Salmon)

MultiQC can have several rows per sample (one per tool) → merge them per cell.

In [7]:
stats = pd.read_csv(f"{RNA_RES}/multiqc/multiqc_data/multiqc_general_stats.txt", sep="\t", index_col=0)
stats.index = cell_from_rna(stats.index)
stats = stats[pd.notna(stats.index)]           # drop rows that aren't cells
stats = stats.groupby(level=0).first()         # merge rows of the same cell, column by column

print(stats.shape[0], "cells with RNA stats")
display(stats.head(3))

48 cells with RNA stats


,salmon-percent_mapped,salmon-num_mapped,salmon-library_types,salmon-compatible_fragment_ratio,salmon-strand_mapping_bias,star-total_reads,star-mapped,star-mapped_percent,star-uniquely_mapped,star-uniquely_mapped_percent,star-multimapped,fastp-pct_duplication,fastp-after_filtering_q30_rate,fastp-after_filtering_q30_bases,fastp-filtering_result_passed_filter_reads,fastp-after_filtering_gc_content,fastp-pct_surviving
cell_00,42.384304,0.343838,IU,100.0,0.387760,0.811239,0.442774,54.58,0.387638,47.78,0.055136,37.0981,85.4482,189.044052,1.622478,39.0234,62.694869
cell_01,NaN,NaN,None,NaN,NaN,0.225442,0.005762,2.56,0.000147,0.07,0.005615,41.3803,83.0896,43.242147,0.450884,41.3986,60.222735
cell_02,0.006199,0.000026,IU,100.0,0.318182,0.419433,0.016354,3.90,0.003088,0.74,0.013266,21.1897,63.5200,68.513662,0.838866,40.1369,41.380811


## 1.3 Build AnnData

- cells = all 48 cells of the plate map; cells missing from a matrix are filled with 0 (flagged in `rna_in_htseq` / `rna_in_salmon`)
- genes = genes present in both HTSeq and Salmon
- `X` = HTSeq raw counts; every matrix also stored in `.layers`

In [8]:
cells = plate.index                                    # cell_00 ... cell_47
genes = htseq.columns.intersection(salmon_counts.columns)

print(f"{len(cells)} cells | HTSeq: {htseq.shape[0]} | Salmon: {salmon_counts.shape[0]}")
print("missing from HTSeq :", sorted(set(cells) - set(htseq.index)))
print("missing from Salmon:", sorted(set(cells) - set(salmon_counts.index)))
print(f"{len(genes)} genes shared (HTSeq {htseq.shape[1]}, Salmon {salmon_counts.shape[1]})")


def aligned(m):
    """Same cells and genes, same order, for every matrix; missing cells -> 0."""
    return m.reindex(index=cells, columns=genes, fill_value=0).values.astype("float32")


adata = ad.AnnData(X=aligned(htseq),
                   obs=pd.DataFrame(index=cells),
                   var=pd.DataFrame(index=genes))
adata.layers["htseq_counts"]        = aligned(htseq)
adata.layers["salmon_counts"]       = aligned(salmon_counts)
adata.layers["salmon_tpm"]          = aligned(salmon_tpm)
adata.layers["salmon_lengthscaled"] = aligned(salmon_lengthscaled)

adata.var["gene_symbol"] = gene_table["gene_symbol"].reindex(adata.var_names)

adata.obs = (adata.obs
             .join(plate.add_prefix("lib_"))           # lib_dna_well, lib_rna_well, lib_spatial_well, ...
             .join(stats.add_prefix("rna_")))
adata.obs["rna_in_htseq"]  = adata.obs_names.isin(htseq.index)
adata.obs["rna_in_salmon"] = adata.obs_names.isin(salmon_counts.index)
adata

48 cells | HTSeq: 43 | Salmon: 34
missing from HTSeq : ['cell_07', 'cell_08', 'cell_11', 'cell_32', 'cell_43']
missing from Salmon: ['cell_01', 'cell_03', 'cell_07', 'cell_08', 'cell_11', 'cell_14', 'cell_21', 'cell_24', 'cell_29', 'cell_31', 'cell_32', 'cell_34', 'cell_38', 'cell_45']
10282 genes shared (HTSeq 12899, Salmon 12293)


AnnData object with n_obs × n_vars = 48 × 10282
    obs: 'lib_dna_well', 'lib_dna_index', 'lib_dna_i5', 'lib_dna_i7', 'lib_rna_well', 'lib_rna_index', 'lib_rna_i5', 'lib_rna_i7', 'lib_spatial_well', 'lib_spatial_i5', 'lib_spatial_i7', 'lib_trekker_tile', 'rna_salmon-percent_mapped', 'rna_salmon-num_mapped', 'rna_salmon-library_types', 'rna_salmon-compatible_fragment_ratio', 'rna_salmon-strand_mapping_bias', 'rna_star-total_reads', 'rna_star-mapped', 'rna_star-mapped_percent', 'rna_star-uniquely_mapped', 'rna_star-uniquely_mapped_percent', 'rna_star-multimapped', 'rna_fastp-pct_duplication', 'rna_fastp-after_filtering_q30_rate', 'rna_fastp-after_filtering_q30_bases', 'rna_fastp-filtering_result_passed_filter_reads', 'rna_fastp-after_filtering_gc_content', 'rna_fastp-pct_surviving', 'rna_in_htseq', 'rna_in_salmon'
    var: 'gene_symbol'
    layers: 'htseq_counts', 'salmon_counts', 'salmon_tpm', 'salmon_lengthscaled'

## 1.4 Extra RNA metadata: exonic/intergenic, mitochondrial, cell type, cell cycle

In [9]:
# Qualimap % exonic / intergenic
qm = pd.read_csv(f"{RNA_RES}/secondary_analyses/secondary_metrics/pipeline_metrics_summary_percents.csv", index_col=0)
qm.index = cell_from_rna(qm.index)

# mitochondrial counts and proportion
mt = pd.read_csv(f"{HTSEQ}/df_mt_gene_counts_starhtseq.tsv", sep="\t", index_col=0)
mt.index = cell_from_rna(mt.index)

# SingleR cell type + cell cycle phase
ct = pd.read_csv(f"{RNA_RES}/tertiary_analyses/df_cell_typing_summary_singler_hpca_gtex_tcga.tsv", sep="\t", index_col=0)
ct.index = cell_from_rna(ct.index)

for name, t in [("qm", qm), ("mt", mt), ("ct", ct)]:
    print(name, t.shape, t.columns.tolist())

adata.obs = (adata.obs
             .join(qm.add_prefix("rna_qm_"))
             .join(mt.add_prefix("rna_mt_"))
             .join(ct.add_prefix("rna_ct_"))
             .join(htseq_summary.add_prefix("rna_htseq")))   # -> rna_htseq__no_feature, ...
display(adata.obs.filter(regex="^rna_(qm|mt|ct)_").head(3))

qm (47, 4) ['total.alignments', 'reads.aligned_P_Total', 'exonic_P_gen', 'intergenic_P_gen']
mt (43, 5) ['TotalFeatures', 'MT_NumFeatures', 'MT_Counts', 'Total_Counts', 'PropMT']
ct (18, 5) ['Phase', 'Progenitor', 'Tissue', 'TGCA_Tissue', 'TGCA_Tumor']


,rna_qm_total.alignments,rna_qm_reads.aligned_P_Total,rna_qm_exonic_P_gen,rna_qm_intergenic_P_gen,rna_mt_TotalFeatures,rna_mt_MT_NumFeatures,rna_mt_MT_Counts,rna_mt_Total_Counts,rna_mt_PropMT,rna_ct_Phase,rna_ct_Progenitor,rna_ct_Tissue,rna_ct_TGCA_Tissue,rna_ct_TGCA_Tumor
cell_id,,,,,,,,,,,,,,
cell_00,1488362.0,54.574669,79.794271,6.460979,153.0,14.0,139702.0,294485.0,0.474394,NaN,NaN,NaN,NaN,NaN
cell_01,78441.0,2.531250,0.947867,49.289100,1.0,0.0,0.0,1.0,0.000000,NaN,NaN,NaN,NaN,NaN
cell_02,188570.0,3.898835,0.878033,11.749681,36.0,1.0,1.0,92.0,0.010870,NaN,NaN,NaN,NaN,NaN


## 1.5 RNA QC → `rna_pass`

BioSkryb ClusterQC (BJ-Expression manual): alignment >70% = cat 5, 50–70% = cat 4, 30–50% = cat 3, 10–30% = cat 2.
Our cutoffs are a bit looser (keep cat-3-like cells); tighten `MIN_UNIQ` to 50 to follow BioSkryb strictly.

In [10]:
COL_UNIQ  = "rna_star-uniquely_mapped_percent"                 # STAR % uniquely mapped
COL_READS = "rna_fastp-filtering_result_passed_filter_reads"   # fastp reads passing filter (MILLIONS)
COL_DUP   = "rna_fastp-pct_duplication"                        # fastp % duplication

MIN_UNIQ  = 25     # % uniquely aligned
MIN_READS = 0.5    # million reads after filtering
MAX_DUP   = 40     # % duplication

obs = adata.obs
display(obs[[COL_UNIQ, COL_READS, COL_DUP]].describe().round(2))

rna_checks = pd.DataFrame({
    "uniq":      obs[COL_UNIQ]  >= MIN_UNIQ,
    "reads":     obs[COL_READS] >= MIN_READS,
    "dup":       obs[COL_DUP]   <= MAX_DUP,
    "in_htseq":  obs["rna_in_htseq"],
    "in_salmon": obs["rna_in_salmon"],
})                                             # NaN -> False, i.e. missing values fail
print(rna_checks.sum(), "\n")                  # cells passing each check

adata.obs["rna_pass"] = rna_checks.all(axis=1)
adata.obs["rna_failed_on"] = rna_checks.apply(lambda r: ",".join(r.index[~r]) or "-", axis=1)

print(adata.obs["rna_pass"].sum(), "of", adata.n_obs, "cells pass RNA QC")
display(adata.obs[["lib_rna_well", COL_UNIQ, COL_READS, COL_DUP, "rna_pass", "rna_failed_on", "rna_ct_Progenitor"]]
        .sort_values(COL_UNIQ, ascending=False).round(2))

,rna_star-uniquely_mapped_percent,rna_fastp-filtering_result_passed_filter_reads,rna_fastp-pct_duplication
count,47.00,48.00,48.00
mean,20.81,2.77,33.19
std,26.76,5.84,11.22
min,0.04,0.00,14.67
25%,0.09,0.33,23.27
50%,0.15,0.84,34.25
75%,39.11,2.67,43.01
max,76.43,37.80,55.96


uniq         19
reads        32
dup          33
in_htseq     43
in_salmon    34
dtype: int64 

18 of 48 cells pass RNA QC


,lib_rna_well,rna_star-uniquely_mapped_percent,rna_fastp-filtering_result_passed_filter_reads,rna_fastp-pct_duplication,rna_pass,rna_failed_on,rna_ct_Progenitor
cell_id,,,,,,,
cell_10,K7,76.43,6.14,28.18,True,-,Monocyte
cell_22,G8,73.90,37.80,30.48,True,-,Gametocytes
cell_41,J9,72.82,3.56,21.98,True,-,Neurons
cell_12,M7,72.43,2.67,24.18,True,-,Macrophage
cell_19,D8,68.88,12.66,25.39,True,-,Astrocyte
cell_42,K9,67.86,6.54,20.83,True,-,Neurons
cell_13,N7,52.63,0.73,18.93,True,-,Astrocyte
cell_25,J8,51.55,7.42,31.28,True,-,Chondrocytes
cell_35,D9,49.44,1.11,35.93,True,-,NaN


**Still to use from the RNA results:** BAM files (`secondary_analyses/alignment_htseq/`) for TAF1 / IGV,
transcript-level Salmon matrices.

# 2. DNA (BJ-DNA-QC — subsampled, so QC only)

## 2.1 Metrics and BioSkryb score

In [11]:
# all metrics, one row per DNA library -> renamed to the cell it belongs to
dna = pd.read_csv(f"{DNA_QC}/nf-preseq-pipeline_all_metrics_mqc.txt", sep="\t", index_col=0)
dna.index = cell_from_dna(dna.index)
dna = dna[pd.notna(dna.index)]
dna = dna.reindex(adata.obs_names)             # same cells as adata; NaN where no DNA

# BioSkryb score: CompositeScore = number of passed checks (0-5), NOT the quadrant category.
# Careful: empty wells can score high (MAPD etc. look 'fine' with no reads).
bsk = pd.read_csv(f"{DNA_QC}/DNA-QC_ConsensusScores.txt", sep="\t").set_index("SampleId")
bsk.index = cell_from_dna(bsk.index)
bsk = bsk[pd.notna(bsk.index)].reindex(adata.obs_names)

print(dna.notna().any(axis=1).sum(), "cells with DNA metrics |", dna.shape[1], "metrics")
display(bsk.head(3))
print(bsk["CompositeScore"].value_counts(dropna=False).sort_index())

47 cells with DNA metrics | 259 metrics


,CompositeScore,Group,Verdict_preseq_count,Verdict_PCT_CHIMERAS,Verdict_chrM,Verdict_MAPD_CNV_Log2,Verdict_SKEW_CNV
cell_id,,,,,,,
cell_00,2.0,Group1,0.0,0.0,1.0,0.0,1.0
cell_01,3.0,Group1,0.0,1.0,1.0,0.0,1.0
cell_02,3.0,Group1,0.0,1.0,1.0,0.0,1.0


CompositeScore
2.0    17
3.0    28
4.0     2
NaN     1
Name: count, dtype: int64


## 2.2 DNA QC → `dna_pass`

`insert` and `reads` remove empty wells (primer dimers, ~23 bp inserts) whose MAPD = 0 looks 'perfect'.
The BioSkryb score is kept as information only: its Preseq / MAPD verdicts fail even good libraries.

In [12]:
dna_checks = pd.DataFrame({
    "insert":   dna["MEDIAN_INSERT_SIZE"]  > 250,    # real library, not primer dimers (~23 bp)
    "reads":    dna["READ_PAIRS_EXAMINED"] >= 1e6,   # enough usable pairs (good cells ~2.4M)
    "mapd":     dna["MAPD.Score"]          < 0.6,    # smooth coverage (good cells 0.36-0.54)
    "segmad":   dna["cnv_segment_MAD"]     < 1.0,    # low noise within CNV segments
    # "bioskryb": bsk["CompositeScore"]    >= 3,     # off: fails good libraries (see above)
})
print(dna_checks.sum(), "\n")

dna_pass      = dna_checks.all(axis=1)
dna_failed_on = dna_checks.apply(lambda r: ",".join(r.index[~r]) or "-", axis=1)

print("DNA pass:", sorted(dna_pass[dna_pass].index))

display(pd.crosstab(adata.obs["rna_pass"], dna_pass, rownames=["RNA pass"], colnames=["DNA pass"]))
display(dna[["MEDIAN_INSERT_SIZE", "READ_PAIRS_EXAMINED", "MAPD.Score", "cnv_segment_MAD"]]
        .join(bsk.filter(regex="^(CompositeScore|Verdict_)"))
        .assign(dna_well=adata.obs["lib_dna_well"], rna_pass=adata.obs["rna_pass"],
                dna_pass=dna_pass, dna_failed_on=dna_failed_on)
        .sort_values("CompositeScore", ascending=False))

insert    23
reads     21
mapd      44
segmad    35
dtype: int64 

DNA pass: ['cell_00', 'cell_02', 'cell_05', 'cell_06', 'cell_10', 'cell_12', 'cell_13', 'cell_16', 'cell_17', 'cell_19', 'cell_22', 'cell_23', 'cell_25', 'cell_32', 'cell_37', 'cell_39', 'cell_40', 'cell_41', 'cell_42']


DNA pass,False,True
RNA pass,,
False,27,3
True,2,16


,MEDIAN_INSERT_SIZE,READ_PAIRS_EXAMINED,MAPD.Score,cnv_segment_MAD,CompositeScore,Verdict_preseq_count,Verdict_PCT_CHIMERAS,Verdict_chrM,Verdict_MAPD_CNV_Log2,Verdict_SKEW_CNV,dna_well,rna_pass,dna_pass,dna_failed_on
cell_id,,,,,,,,,,,,,,
cell_10,378.0,2452498.0,0.356989,3.916678e-01,4.0,0.0,1.0,1.0,1.0,1.0,K2,True,True,-
cell_30,24.0,7516.0,0.111111,1.695112e-03,4.0,0.0,1.0,1.0,1.0,1.0,O3,False,False,"insert,reads"
cell_02,334.0,2386669.0,0.508763,7.274550e-01,3.0,0.0,1.0,1.0,0.0,1.0,C2,False,True,-
cell_11,23.0,32345.0,0.393278,2.235211e+00,3.0,0.0,1.0,1.0,0.0,1.0,L2,False,False,"insert,reads,segmad"
cell_09,23.0,29561.0,0.311182,2.082230e+00,3.0,0.0,1.0,1.0,0.0,1.0,J2,False,False,"insert,reads,segmad"
cell_08,24.0,56238.0,0.329910,1.986803e+00,3.0,0.0,1.0,1.0,0.0,1.0,I2,False,False,"insert,reads,segmad"
cell_01,22.0,12564.0,0.109399,5.810953e-01,3.0,0.0,1.0,1.0,0.0,1.0,B2,False,False,"insert,reads"
cell_13,369.0,2428694.0,0.448179,5.216418e-01,3.0,0.0,1.0,1.0,0.0,1.0,N2,True,True,-
cell_12,386.0,2439803.0,0.398341,5.981144e-01,3.0,0.0,1.0,1.0,0.0,1.0,M2,True,True,-


## 2.3 Add DNA info to `adata.obs`

Keep one version of each metric: drop the non-deduplicated copies (`_ndd`), merge leftovers
(`index.1`, `Unnamed…`) and text label columns.

Note: `dna_total_reads` = full raw depth (before subsampling); `dna_TOTAL_READS` = subsampled reads used for QC.

In [13]:
junk = [c for c in dna.columns
        if c.endswith("_ndd") or "_ndd." in c
        or c.startswith(("index", "Unnamed", "LIBRARY"))
        or c in {"SAMPLE", "READ_GROUP", "CATEGORY", "PAIR_ORIENTATION"}]
dna_clean = dna.drop(columns=junk)

# remove DNA columns from a previous run, so this cell can be re-run
adata.obs = adata.obs.drop(columns=[c for c in adata.obs.columns if c.startswith("dna_")])

adata.obs = (adata.obs
             .join(dna_clean.add_prefix("dna_"))
             .join(bsk.filter(like="Verdict_").add_prefix("dna_bioskryb_")))
adata.obs["dna_bioskryb_score"] = bsk["CompositeScore"]
adata.obs["dna_pass"]           = dna_pass
adata.obs["dna_failed_on"]      = dna_failed_on

print(f"{dna.shape[1]} DNA metrics -> {dna_clean.shape[1]} kept")
display(adata.obs.filter(like="dna_").head(3))

259 DNA metrics -> 135 kept


,lib_dna_well,lib_dna_index,lib_dna_i5,lib_dna_i7,dna_TOTAL_READS,dna_PF_READS,dna_PCT_PF_READS,dna_PF_NOISE_READS,dna_PF_READS_ALIGNED,dna_PCT_PF_READS_ALIGNED,...,dna_MAPD_CNV_Log2,dna_SKEW_CNV,dna_bioskryb_Verdict_preseq_count,dna_bioskryb_Verdict_PCT_CHIMERAS,dna_bioskryb_Verdict_chrM,dna_bioskryb_Verdict_MAPD_CNV_Log2,dna_bioskryb_Verdict_SKEW_CNV,dna_bioskryb_score,dna_pass,dna_failed_on
cell_id,,,,,,,,,,,,,,,,,,,,,
cell_00,A2,C-01A,AGACCCTATA,TGGCAGGGCG,4608814.0,4608814.0,1.0,0.0,4607249.0,0.999660,...,0.344548,0.170893,0.0,0.0,1.0,0.0,1.0,2.0,True,-
cell_01,B2,C-02A,TAACTGAATC,AAAGGCTAGA,8206.0,8206.0,1.0,0.0,7996.0,0.974409,...,1.427046,0.237891,0.0,1.0,1.0,0.0,1.0,3.0,False,"insert,reads"
cell_02,C2,C-01B,AGTACAGAAT,GAATTATATC,4553464.0,4553464.0,1.0,0.0,4552370.0,0.999760,...,0.380352,0.148257,0.0,1.0,1.0,0.0,1.0,3.0,True,-


# 3. Combined verdict

In [14]:
adata.obs["both_pass"] = adata.obs["rna_pass"] & adata.obs["dna_pass"]
adata.obs["libs_pass"] = np.select(
    [adata.obs["both_pass"], adata.obs["rna_pass"], adata.obs["dna_pass"]],
    ["both", "rna_only", "dna_only"],
    default="none")
print(adata.obs["libs_pass"].value_counts())

# DNA wells of the cells passing both (e.g. to select samples for BJ-WGS in BaseJumper)
print("DNA wells, both pass:", sorted(adata.obs.loc[adata.obs["both_pass"], "lib_dna_well"]))
print("DNA wells, DNA pass:", sorted(adata.obs.loc[adata.obs["dna_pass"], "lib_dna_well"]))

libs_pass
none        27
both        16
dna_only     3
rna_only     2
Name: count, dtype: int64
DNA wells, both pass: ['A2', 'A3', 'B3', 'D3', 'F4', 'G2', 'G3', 'H3', 'H4', 'I4', 'J3', 'J4', 'K2', 'K4', 'M2', 'N2']
DNA wells, DNA pass: ['A2', 'A3', 'A4', 'B3', 'C2', 'D3', 'F2', 'F4', 'G2', 'G3', 'H3', 'H4', 'I4', 'J3', 'J4', 'K2', 'K4', 'M2', 'N2']


# 4. Cell types with MapMyCells (Allen basal ganglia atlas)

- query = all **HTSeq genes** (not only the HTSeq ∩ Salmon genes in `adata`), so more marker genes are usable
- gene IDs without Ensembl version (the atlas uses `ENSG…` without `.version`)
- CPU implementation forced: the GPU path of `cell_type_mapper` 1.5.2 crashes (numpy on a CUDA tensor)

In [15]:
from cell_type_mapper.cli.from_specified_markers import FromSpecifiedMarkersRunner
import cell_type_mapper.type_assignment.election_runner as election_runner
election_runner.use_torch = lambda: False       # force CPU implementation

os.makedirs(MMC_DIR, exist_ok=True)
query_h5ad = f"{MMC_DIR}/query.h5ad"
out_csv    = f"{MMC_DIR}/mmc_results.csv"
out_json   = f"{MMC_DIR}/mmc_results.json"

# raw counts, all cells x all HTSeq genes, gene IDs without version
counts = htseq.reindex(adata.obs_names).fillna(0)
counts.columns = counts.columns.str.split(".").str[0]
counts = counts.T.groupby(level=0).sum().T      # merge IDs that collide after removing versions

markers = json.load(open(MMC_MARKERS))
marker_genes = {g for v in markers.values() if isinstance(v, list) for g in v}
print(f"{len(marker_genes & set(counts.columns))} of {len(marker_genes)} marker genes in the query")

query = ad.AnnData(X=counts.values.astype("float32"),
                   obs=pd.DataFrame(index=counts.index),
                   var=pd.DataFrame(index=counts.columns))
query.write_h5ad(query_h5ad)

config = {
    "query_path": query_h5ad,
    "extended_result_path": out_json,
    "csv_result_path": out_csv,
    "tmp_dir": tempfile.mkdtemp(),
    "cloud_safe": False,
    "flatten": False,                                  # use the full taxonomy hierarchy
    "query_markers":     {"serialized_lookup": MMC_MARKERS},
    "precomputed_stats": {"path": MMC_STATS},
    "type_assignment": {
        "normalization": "raw",                        # raw counts; the mapper converts to log2CPM
        "bootstrap_iteration": 100,
        "bootstrap_factor": 0.5,
        "n_processors": 4,
        "chunk_size": 1000,
    },
}
FromSpecifiedMarkersRunner(args=[], input_data=config).run()

3847 of 6733 marker genes in the query
=== Running Hierarchical Mapping 1.5.2 with config ===
{
  "query_markers": {
    "log_level": "ERROR",
    "collapse_markers": false,
    "serialized_lookup": "/home/gdallagl/myworkdir/XDP/data/AllenAtlas/BGT_human_20250507/Human.query_markers.20250507.json"
  },
  "summary_metadata_path": null,
  "extended_result_path": "/home/gdallagl/myworkdir/XDP/data/_tmp/G_and_T/mapmycells/mmc_results.json",
  "obsm_key": null,
  "flatten": false,
  "precomputed_stats": {
    "log_level": "ERROR",
    "path": "/home/gdallagl/myworkdir/XDP/data/AllenAtlas/BGT_human_20250507/Human.precomputed_stats.20250507.h5"
  },
  "extended_result_dir": null,
  "log_path": null,
  "query_path": "/home/gdallagl/myworkdir/XDP/data/_tmp/G_and_T/mapmycells/query.h5ad",
  "csv_result_path": "/home/gdallagl/myworkdir/XDP/data/_tmp/G_and_T/mapmycells/mmc_results.csv",
  "drop_level": null,
  "nodes_to_drop": null,
  "cloud_safe": false,
  "tmp_dir": "/tmp/tmpa_r2sp3f",
  "hdf5_r

/home/gdallagl/myworkdir/XDP/.venv/lib/python3.11/site-packages/cell_type_mapper/cli/cli_log.py:73: UserWarning: numpy's internal parallelization is enabled. This could cause independent worker processes to compete for resources, degrading performance. We recommend setting the following environment variables to '1' to improve performance
{
  "NUMEXPR_NUM_THREADS": "",
  "MKL_NUM_THREADS": "",
  "OMP_NUM_THREADS": ""
}
  warnings.warn(msg)
/home/gdallagl/myworkdir/XDP/.venv/lib/python3.11/site-packages/cell_type_mapper/cli/cli_log.py:104: FutureWarning: `__version__` is deprecated, use `importlib.metadata.version('anndata')` instead.
  self.env(f"anndata version: {anndata.__version__}")


FILE TRACKER: copied ../Human.precomputed_stats.20250507.h5 to ../Human.precomputed_stats.20250507_4b2uw98w.h5
BENCHMARK: spent 3.9719e+00 seconds validating config and copying data
using ../Human.precomputed_stats.20250507_4b2uw98w.h5 for precomputed_stats
reading taxonomy_tree from ../Human.precomputed_stats.20250507_4b2uw98w.h5


/home/gdallagl/myworkdir/XDP/.venv/lib/python3.11/site-packages/cell_type_mapper/taxonomy/utils.py:253: UserWarning: This taxonomy has no mapping from leaf_node -> rows in the cell by gene matrix
  warnings.warn("This taxonomy has no mapping from leaf_node -> rows "
/home/gdallagl/myworkdir/XDP/.venv/lib/python3.11/site-packages/cell_type_mapper/cli/cli_log.py:73: UserWarning: 2886 marker genes were not present in the query dataset. They have been ignored
  warnings.warn(msg)


BENCHMARK: spent 1.1814e-01 seconds creating query marker cache
Running CPU implementation of type assignment.
BENCHMARK: spent 2.2421e+01 seconds assigning cell types
Writing marker genes to output file
FILE TRACKER: cleaning up ../file_tracker_i8zx2nwb
MAPPING FROM SPECIFIED MARKERS RAN SUCCESSFULLY
CLEANING UP


In [16]:
mmc = pd.read_csv(out_csv, comment="#", index_col=0)    # lines starting with '#' are metadata

adata.obs = adata.obs.drop(columns=[c for c in adata.obs if c.startswith("mmc_")])   # re-runnable
adata.obs = adata.obs.join(mmc.add_prefix("mmc_"))
display(adata.obs.loc[adata.obs["rna_pass"]].filter(regex="^mmc_.*(name|probability)"))

,mmc_Neighborhood_name,mmc_Neighborhood_bootstrapping_probability,mmc_Class_name,mmc_Class_bootstrapping_probability,mmc_Subclass_name,mmc_Subclass_bootstrapping_probability,mmc_Group_name,mmc_Group_bootstrapping_probability,mmc_Cluster_name,mmc_Cluster_bootstrapping_probability
cell_id,,,,,,,,,,
cell_00,Nonneuron,1.00,Immune,0.56,Microglia,0.42,Microglia,1.00,Human-540,0.68
cell_06,Nonneuron,0.95,OPC-Oligo,0.62,Oligodendrocyte,0.79,ImOligo,0.64,Human-49,0.40
cell_10,Nonneuron,1.00,OPC-Oligo,1.00,Oligodendrocyte,1.00,ImOligo,0.54,Human-277,0.39
cell_12,Nonneuron,1.00,Astro-Epen,1.00,Astrocyte,1.00,ImAstro,0.93,Human-210,0.97
cell_13,Subpallium GABA,1.00,CN LGE GABA,1.00,STR D1 MSN,0.94,STRd D1 Striosome MSN,0.92,Human-87,0.41
cell_16,Subpallium GABA,0.95,CN LGE GABA,0.85,STR D2 MSN,0.95,STRd D2 Matrix MSN,1.00,Human-538,0.61
cell_17,Subpallium GABA,1.00,CN LGE GABA,1.00,STR D1 MSN,0.86,STRd D1 Striosome MSN,0.45,Human-514,0.49
cell_19,Nonneuron,1.00,OPC-Oligo,1.00,Oligodendrocyte,0.99,Oligo OPALIN,0.86,Human-1,0.98
cell_22,Nonneuron,1.00,OPC-Oligo,1.00,Oligodendrocyte,1.00,Oligo OPALIN,0.77,Human-1,0.99


# 5. Final AnnData

## 5.1 Most important columns first

In [17]:
first = [
    "mmc_Group_name",
    # cell identity: which wells hold this cell's libraries
    "lib_dna_well", "lib_rna_well", "lib_spatial_well",
    # verdicts
    "libs_pass", "both_pass", "rna_pass", "rna_failed_on", "dna_pass", "dna_failed_on", "dna_bioskryb_score",
    # RNA key QC
    "rna_star-uniquely_mapped_percent", "rna_star-mapped_percent",
    "rna_fastp-filtering_result_passed_filter_reads", "rna_fastp-pct_duplication",
    "rna_salmon-percent_mapped", "rna_qm_exonic_P_gen", "rna_qm_intergenic_P_gen",
    "rna_mt_PropMT", "rna_mt_TotalFeatures", "rna_in_htseq", "rna_in_salmon",
    # RNA biology
    "rna_ct_Phase", "rna_ct_Tissue", "rna_ct_Progenitor",
    # DNA key QC
    "dna_total_reads", "dna_READ_PAIRS_EXAMINED", "dna_MEDIAN_INSERT_SIZE", "dna_PERCENT_DUPLICATION",
    "dna_MAPD.Score", "dna_cnv_segment_MAD", "dna_cnv_number_of_segments", "dna_PCT_CHIMERAS",
    "dna_preseq_count", "dna_gini_coefficient_index", "dna_chrX", "dna_chrY", "dna_chrM",
]
first = [c for c in first if c in adata.obs.columns]          # skip any that don't exist
adata.obs = adata.obs[first + [c for c in adata.obs.columns if c not in first]]

display(adata.obs[first].sort_values(["both_pass", "rna_pass"], ascending=False))

,mmc_Group_name,lib_dna_well,lib_rna_well,lib_spatial_well,libs_pass,both_pass,rna_pass,rna_failed_on,dna_pass,dna_failed_on,...,dna_PERCENT_DUPLICATION,dna_MAPD.Score,dna_cnv_segment_MAD,dna_cnv_number_of_segments,dna_PCT_CHIMERAS,dna_preseq_count,dna_gini_coefficient_index,dna_chrX,dna_chrY,dna_chrM
cell_id,,,,,,,,,,,,,,,,,,,,,
cell_00,Microglia,A2,A7,A12,both,True,True,-,True,-,...,0.013295,0.383682,5.647328e-01,32.0,0.242067,2.463034e+09,0.228324,0.027758,0.016531,1.258729e-03
cell_06,ImOligo,G2,G7,G12,both,True,True,-,True,-,...,0.008239,0.389519,5.466445e-01,31.0,0.202722,2.303114e+09,0.217925,0.018872,0.014265,9.272756e-04
cell_10,ImOligo,K2,K7,K12,both,True,True,-,True,-,...,0.007982,0.356989,3.916678e-01,21.0,0.108018,3.373218e+09,0.110084,0.027639,0.010909,1.235116e-04
cell_12,ImAstro,M2,M7,M12,both,True,True,-,True,-,...,0.017696,0.398341,5.981144e-01,14.0,0.110543,3.449418e+09,0.105694,0.022996,0.008589,4.394552e-04
cell_13,STRd D1 Striosome MSN,N2,N7,N12,both,True,True,-,True,-,...,0.007212,0.448179,5.216418e-01,13.0,0.131664,3.376357e+09,0.134660,0.029331,0.010870,1.489679e-04
cell_16,STRd D2 Matrix MSN,A3,A8,A13,both,True,True,-,True,-,...,0.011883,0.497869,4.972173e-01,27.0,0.104006,2.966753e+09,0.141254,0.031670,0.008400,3.445243e-04
cell_17,STRd D1 Striosome MSN,B3,B8,B13,both,True,True,-,True,-,...,0.009416,0.479979,4.773243e-01,18.0,0.112168,3.251824e+09,0.149770,0.030822,0.011188,2.188959e-04
cell_19,Oligo OPALIN,D3,D8,D13,both,True,True,-,True,-,...,0.003994,0.404276,5.696675e-01,9.0,0.080302,3.390636e+09,0.114261,0.027652,0.009275,4.092906e-04
cell_22,Oligo OPALIN,G3,G8,G13,both,True,True,-,True,-,...,0.010045,0.406209,7.238714e-01,28.0,0.090133,3.191569e+09,0.128585,0.019652,0.010906,2.834942e-04


In [18]:
adata.uns["plate_map"] = "WALKUP-21131 plate map: cell_id -> DNA / RNA / spatial wells (lib_* columns)"
display(adata)
display(adata.var.head())
print(adata.obs.columns.tolist())

AnnData object with n_obs × n_vars = 48 × 10282
    obs: 'mmc_Group_name', 'lib_dna_well', 'lib_rna_well', 'lib_spatial_well', 'libs_pass', 'both_pass', 'rna_pass', 'rna_failed_on', 'dna_pass', 'dna_failed_on', 'dna_bioskryb_score', 'rna_star-uniquely_mapped_percent', 'rna_star-mapped_percent', 'rna_fastp-filtering_result_passed_filter_reads', 'rna_fastp-pct_duplication', 'rna_salmon-percent_mapped', 'rna_qm_exonic_P_gen', 'rna_qm_intergenic_P_gen', 'rna_mt_PropMT', 'rna_mt_TotalFeatures', 'rna_in_htseq', 'rna_in_salmon', 'rna_ct_Phase', 'rna_ct_Tissue', 'rna_ct_Progenitor', 'dna_total_reads', 'dna_READ_PAIRS_EXAMINED', 'dna_MEDIAN_INSERT_SIZE', 'dna_PERCENT_DUPLICATION', 'dna_MAPD.Score', 'dna_cnv_segment_MAD', 'dna_cnv_number_of_segments', 'dna_PCT_CHIMERAS', 'dna_preseq_count', 'dna_gini_coefficient_index', 'dna_chrX', 'dna_chrY', 'dna_chrM', 'lib_dna_index', 'lib_dna_i5', 'lib_dna_i7', 'lib_rna_index', 'lib_rna_i5', 'lib_rna_i7', 'lib_spatial_i5', 'lib_spatial_i7', 'lib_trekker_til

,gene_symbol
ENSG00000000419.14,DPM1
ENSG00000000457.14,SCYL3
ENSG00000000971.16,CFH
ENSG00000001084.13,GCLC
ENSG00000001460.18,STPG1


['mmc_Group_name', 'lib_dna_well', 'lib_rna_well', 'lib_spatial_well', 'libs_pass', 'both_pass', 'rna_pass', 'rna_failed_on', 'dna_pass', 'dna_failed_on', 'dna_bioskryb_score', 'rna_star-uniquely_mapped_percent', 'rna_star-mapped_percent', 'rna_fastp-filtering_result_passed_filter_reads', 'rna_fastp-pct_duplication', 'rna_salmon-percent_mapped', 'rna_qm_exonic_P_gen', 'rna_qm_intergenic_P_gen', 'rna_mt_PropMT', 'rna_mt_TotalFeatures', 'rna_in_htseq', 'rna_in_salmon', 'rna_ct_Phase', 'rna_ct_Tissue', 'rna_ct_Progenitor', 'dna_total_reads', 'dna_READ_PAIRS_EXAMINED', 'dna_MEDIAN_INSERT_SIZE', 'dna_PERCENT_DUPLICATION', 'dna_MAPD.Score', 'dna_cnv_segment_MAD', 'dna_cnv_number_of_segments', 'dna_PCT_CHIMERAS', 'dna_preseq_count', 'dna_gini_coefficient_index', 'dna_chrX', 'dna_chrY', 'dna_chrM', 'lib_dna_index', 'lib_dna_i5', 'lib_dna_i7', 'lib_rna_index', 'lib_rna_i5', 'lib_rna_i7', 'lib_spatial_i5', 'lib_spatial_i7', 'lib_trekker_tile', 'rna_salmon-num_mapped', 'rna_salmon-library_types',

# 6. Save

In [19]:
# h5ad can't store object columns with mixed types -> make them plain strings
for c in adata.obs.select_dtypes(include="object").columns:
    adata.obs[c] = adata.obs[c].astype(str)
for c in adata.var.select_dtypes(include="object").columns:
    adata.var[c] = adata.var[c].astype(str)

adata.write_h5ad(OUT_H5AD)
print("saved:", OUT_H5AD)

saved: /home/gdallagl/myworkdir/XDP/data/XDP/G_and_T/rna_libs/adata_RNA_DNA-QC.h5ad
